In [ ]:
!pip install --upgrade transformers -q
!pip install faiss-cpu -q
!pip install  datasets sentence-transformers -q

In [1]:
# load the Algerian legal text corpus
from datasets import load_dataset
DATASET_NAME = "hatemestinbejaia/STCALIR-Dataset"
dataset = load_dataset(DATASET_NAME, 'collection')
print(dataset)
collection = dataset["train"]
collection_df = collection.to_pandas()
print(collection_df.head())
collection_df.info()
collection_df[["docid", "text"]].head()

README.md:   0%|          | 0.00/965 [00:00<?, ?B/s]

collection/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 27.9MB            

collection/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/105201 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['docid', 'text'],
        num_rows: 105201
    })
})
                docid                                               text
0  19750103_01_004_p1  المادة الاولى: يصادق على الاتفاق التجاري المبر...
1  19750103_01_004_p2  المادة 2: ينشر هذا الامر في الجريدة الرسمية لل...
2  19750103_01_012_p1  المادة الاولى: إن الحد الاعلى لإصدار القطعة ال...
3  19750103_01_012_p2  المادة 2: ينشر هذا الامر في الجريدة الرسمية لل...
4  19750107_02_093_p1  المادة الاولى: تحدث مراقبة طبية مخصصة لجميع اع...
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105201 entries, 0 to 105200
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   docid   105201 non-null  object
 1   text    105201 non-null  object
dtypes: object(2)
memory usage: 1.6+ MB


,docid,text
0,19750103_01_004_p1,المادة الاولى: يصادق على الاتفاق التجاري المبر...
1,19750103_01_004_p2,المادة 2: ينشر هذا الامر في الجريدة الرسمية لل...
2,19750103_01_012_p1,المادة الاولى: إن الحد الاعلى لإصدار القطعة ال...
3,19750103_01_012_p2,المادة 2: ينشر هذا الامر في الجريدة الرسمية لل...
4,19750107_02_093_p1,المادة الاولى: تحدث مراقبة طبية مخصصة لجميع اع...


In [2]:
# load the bi-encoder 
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
# bi-encoder used for 1-stage
MODEL_NAME = "abdoelsayed/AraDPR"
encoder = SentenceTransformer(MODEL_NAME)

config.json: reconstructing file:   0%|          |  0.00B /   658B            

config.json: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  711MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  711MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

vocab.txt: reconstructing file:   0%|          |  0.00B /  996kB            

vocab.txt: downloading bytes:           |  0.00B            

special_tokens_map.json: reconstructing file:   0%|          |  0.00B /   112B            

special_tokens_map.json: downloading bytes:           |  0.00B            

In [3]:
# load the LLM gemma 4 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "google/gemma-4-E4B-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Model loaded successfully.")

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 16.0GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded successfully.


In [4]:
# llm_query take a passage and output a generated query
passage= collection_df["text"][13339]
print(passage)
def llm_query(PASSAGE):
    messages = [
        {
            "role": "user",
            "content": f"""
                        أنت خبير في استرجاع المعلومات القانونية باللغة العربية.
            
            مهمتك هي توليد استعلام بحث قانوني واحد باللغة العربية انطلاقًا من
            المقطع القانوني المقدم لك.
            
            [المقطع القانوني]
            {PASSAGE}
            [/المقطع القانوني]
            
            اتبع التعليمات التالية بدقة:
            
            1. أنشئ سؤالًا قانونيًا واحدًا فقط يمثل حاجة معلوماتية حقيقية.
            2. يجب أن يكون السؤال قابلًا للإجابة اعتمادًا على المعلومات الواردة
               في المقطع القانوني.
            3. صِغ السؤال بطريقة طبيعية تشبه الطريقة التي قد يستخدمها شخص يبحث
               عن معلومة قانونية في محرك بحث.
            4. ركّز على الحكم أو القاعدة أو الشرط أو الاستثناء أو الحق أو الالتزام
               القانوني الوارد في المقطع.
            5. لا تنسخ جملة من المقطع حرفيًا. أعد صياغة المعلومة في شكل سؤال.
            6. لا تضف أي معلومة غير موجودة أو غير مدعومة في المقطع.
            7. لا تجعل السؤال عامًا جدًا. يجب أن يشير إلى الموضوع القانوني المحدد
               الذي يتناوله المقطع.
            8. استخدم اللغة العربية الفصحى القانونية الواضحة.
            9. يمكن استخدام صيغ مثل:
               - ما هي شروط ...؟
               - ما هي الحالات التي ...؟
               - ما هي الأحكام المتعلقة بـ ...؟
               - متى يجوز ...؟
               - متى يمنع ...؟
               - ما هي الالتزامات المتعلقة بـ ...؟
               - ما هي الاستثناءات من ...؟
            10. لا تركز على البيانات الوصفية للمصدر، مثل رقم الجريدة أو الصفحة
                أو تاريخ النشر أو تاريخ الإمضاء، إلا إذا كانت هذه المعلومات هي
                موضوع المقطع نفسه.
            11. لا تذكر عبارة "وفقًا للنص" أو "حسب المقطع" أو أي إشارة إلى
                عملية التوليد.
            12. لا تقدم إجابة عن السؤال.
            13. أخرج السؤال فقط، دون شرح أو ترقيم أو علامات اقتباس.
            14. اجعل السؤال موجزًا، ويفضل أن يتراوح بين 8 و20 كلمة.
            
            المقطع القانوني:
            {PASSAGE}
            
            السؤال:
    """
        }
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )
    
    input_length = inputs["input_ids"].shape[-1]
    
    generated_tokens = outputs[0][input_length:]
    
    profile = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
    
    return profile
llm_query(passage)

المادة 2: يمنع اي شخص ان يشعل النار داخل الاملاك الغابية الوطنية او على بعد مسافة تقل عن كيلو متر واحد منها, كما يمنع عليه ان ياتي بها إليها. غير انه لا يرخص باستعمال النار إلا في اغراض نفعية بشرط مراعاة التعليمات الواردة في هذا المرسوم. ملخص يتعلق بوقاية الاملاك الغابية الوطنية وماجاورها من الحرائق. تاريخ النشر 14070612.0 رقم الجريدة 07 رقم  42718 طبيعة 04 صفحة 247 تاريخ الإمضاء 14070611


'ما هي القيود المفروضة على إشعال النار بالقرب من الأملاك الغابية الوطنية ومتى يُسمح بذلك؟'

In [5]:
import numpy as np
import faiss
# encode all passages into vector representations using AraDPR, and indexed them using FAISS.
documents = collection_df["text"].tolist()
document_embeddings = encoder.encode(
    documents,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)
np.save(
    "stcalir_document_embeddings.npy",
    document_embeddings
)
print(document_embeddings.shape)
embedding_dim = document_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(
    document_embeddings.astype("float32")
)
print("Number of indexed documents:", faiss_index.ntotal)
faiss.write_index(
    faiss_index,
    "stcalir_faiss.index"
)

Batches:   0%|          | 0/1644 [00:00<?, ?it/s]

(105201, 768)
Number of indexed documents: 105201


In [6]:
# load the index
faiss_index = faiss.read_index("stcalir_faiss.index")

In [7]:
# randomly select 1,000 legal passages, impose that the passage len is more than 100 words and they must be different 
import numpy as np
import random

# Parameters
n_select = 1000
threshold = 0.9
min_words = 100

# Count words for each document
word_counts = collection_df["text"].str.split().str.len()

# Keep only documents with more than 100 words
eligible_ids = np.where(word_counts > min_words)[0]
print(eligible_ids)
print("Documents with >100 words:", len(eligible_ids))

# Randomize eligible documents
rng = np.random.default_rng(42)
candidate_ids = rng.permutation(eligible_ids)

selected_ids = []

for idx in candidate_ids:
    #print(idx)
    if len(selected_ids) >= n_select:
        break

    # Get embedding from FAISS
    query_vector = faiss_index.reconstruct(int(idx)).reshape(1, -1)

    # Check similarity against selected documents
    if selected_ids:

        selected_vectors = np.vstack([
            faiss_index.reconstruct(int(i))
            for i in selected_ids
        ])

        similarities = np.dot(
            selected_vectors,
            query_vector.T
        ).flatten()

        # Reject if too similar to any selected document
        #if np.max(similarities) >= threshold: continue
        if np.mean(np.sort(similarities)[-3:]) >= threshold: continue

    selected_ids.append(idx)

print(f"Selected {len(selected_ids)} texts")
selected_collection = collection_df.iloc[selected_ids][["docid", "text"]].copy()
print(selected_collection.head())

[     4     17     23 ... 105157 105169 105193]
Documents with >100 words: 21521
Selected 1000 texts
                         docid  \
9733      19830201_05_34130_p5   
71435     20040404_20_73934_p8   
59303  20020210_09_70344_p2_p2   
89582  20070121_06_78270_p2_p9   
27796     19940817_52_55575_p6   

                                                    text  
9733   المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...  
71435  المادة 8 : يتداول مجلس الإدارة في كل المسائل ا...  
59303  المادة 2 استخدام المصطلحات لاغراض هذه الاتفاقي...  
89582  10 بعد المادة 11 من الاتفاقية تضاف مادتان جديد...  
27796  المادة 6: تصنف المناصب العليا المذكورة في الما...  


In [8]:
# Each selected passage is provided to Gemma 4, which generates a corresponding legal query in Modern Standard Arabic

generated_queries = []

for i, passage in enumerate(selected_collection["text"]):

    try:
        query = llm_query(passage)
        generated_queries.append(query)

    except Exception as e:
        print(f"Error at passage {i}: {e}")
        generated_queries.append(None)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(selected_collection)}")
# end of querires generation 
selected_collection["query"] = generated_queries
selected_collection.head()

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000


,docid,text,query
9733,19830201_05_34130_p5,المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...,ما هي مدد سداد سعر التنازل عن العقارات السكنية...
71435,20040404_20_73934_p8,المادة 8 : يتداول مجلس الإدارة في كل المسائل ا...,ما هي المسائل التي يتولى مجلس الإدارة دراستها ...
59303,20020210_09_70344_p2_p2,المادة 2 استخدام المصطلحات لاغراض هذه الاتفاقي...,"ما هي الشروط التي تُعرّف بموجب الاتفاقية ""الجم..."
89582,20070121_06_78270_p2_p9,10 بعد المادة 11 من الاتفاقية تضاف مادتان جديد...,ما هي الحالات التي تمنع الدولة من رفض طلب تسلي...
27796,19940817_52_55575_p6,المادة 6: تصنف المناصب العليا المذكورة في الما...,ما هي الشروط المحددة لتصنيف المناصب العليا مثل...


In [9]:
import numpy as np
import pandas as pd

n_negatives = 100
threshold = 0.5

rng = np.random.default_rng(42)

negative_ids_per_positive = []

for i, positive_idx in enumerate(selected_ids):

    # Get the positive vector from FAISS
    positive_vector = faiss_index.reconstruct(int(positive_idx)).reshape(1, -1)

    # Search using FAISS
    similarities, indices = faiss_index.search(
        positive_vector.astype("float32"),
        faiss_index.ntotal
    )

    similarities = similarities[0]
    indices = indices[0]

    # Keep only documents with similarity < 0.4
    eligible_negatives = indices[similarities < threshold]

    # Remove the positive document itself
    eligible_negatives = eligible_negatives[
        eligible_negatives != positive_idx
    ]

    # Randomly select 100
    if len(eligible_negatives) >= n_negatives:
        negative_ids = rng.choice(
            eligible_negatives,
            size=n_negatives,
            replace=False
        )
    else:
        print(
            f"Warning: only {len(eligible_negatives)} "
            f"eligible negatives for positive {positive_idx}"
        )
        negative_ids = eligible_negatives

    negative_ids_per_positive.append(negative_ids)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(selected_ids)}")

Processed 100/1000
Processed 200/1000
Processed 300/1000
Processed 400/1000
Processed 500/1000
Processed 600/1000
Processed 700/1000
Processed 800/1000
Processed 900/1000
Processed 1000/1000


In [10]:
negative_rows = []

for i, positive_idx in enumerate(selected_ids):

    query = selected_collection.iloc[i]["query"]

    for negative_idx in negative_ids_per_positive[i]:

        negative_rows.append({
            "query": query,
            "positive_docid": collection_df.iloc[positive_idx]["docid"],
            "positive_text": collection_df.iloc[positive_idx]["text"],
            "negative_docid": collection_df.iloc[negative_idx]["docid"],
            "negative_text": collection_df.iloc[negative_idx]["text"]
        })

In [11]:
import pandas as pd

negative_df = pd.DataFrame(negative_rows)

print(negative_df.shape)
print(negative_df.head())

(79110, 5)
                                               query        positive_docid  \
0  ما هي مدد سداد سعر التنازل عن العقارات السكنية...  19830201_05_34130_p5   
1  ما هي مدد سداد سعر التنازل عن العقارات السكنية...  19830201_05_34130_p5   
2  ما هي مدد سداد سعر التنازل عن العقارات السكنية...  19830201_05_34130_p5   
3  ما هي مدد سداد سعر التنازل عن العقارات السكنية...  19830201_05_34130_p5   
4  ما هي مدد سداد سعر التنازل عن العقارات السكنية...  19830201_05_34130_p5   

                                       positive_text  \
0  المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...   
1  المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...   
2  المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...   
3  المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...   
4  المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سع...   

             negative_docid                                      negative_text  
0      19990725_49_66078_p8  المادة 8: ينشر هذا المرسوم في الجريدة الرسمية ...  
1  20080824_48_80771_

In [12]:
negative_df.to_csv(
    "stcalir_query_positive_negative_pairs.csv",
    index=False,
    encoding="utf-8-sig"
)

In [19]:
from datasets import load_dataset, Dataset
TRAIN_QUERY_RATIO =0.9
TRAIN_SIZE = 1.0
SEED=42
QUERY_COLUMN = "querytext"
POSITIVE_COLUMN = "passagetext"
NEGATIVE_COLUMN = "negativepassagetext"
SAVE_DATASETS = True
DATASET_OUTPUT_ROOT = "./stcalir_clean_splits"
import os
def build_bi_encoder_datasets(
    csv_path,
    train_query_ratio=0.90,
    train_size=1.0,
    seed=42,
):
    """
    Load, clean, and deterministically split STCALIR query-positive-negative triplets.

    Cleaning:
      1. Keep only query / positive / negative text.
      2. Remove rows with missing required text.
      3. Remove exact duplicate triplets only.

    Splitting:
      - Split unique queries, not rows.
      - Optionally sample a fraction of training queries for size ablations.
    """

    if not (0 < train_query_ratio < 1):
        raise ValueError("train_query_ratio must be between 0 and 1.")
    if not (0 < train_size <= 1):
        raise ValueError("train_size must be in (0, 1].")

    df = pd.read_csv(csv_path)
    print(f"Original CSV shape: {df.shape}")

    required = ["query", "positive_text", "negative_text"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. Available columns: {df.columns.tolist()}"
        )

    # Keep only fields needed by the bi-encoder training objective.
    df = df[required].copy()

    # Remove missing/empty text fields.
    before = len(df)
    for col in required:
        df[col] = df[col].astype("string").str.strip()
    df = df.replace({"": pd.NA})
    df = df.dropna(subset=required).copy()
    print(f"Rows removed for missing/empty fields: {before - len(df)}")

    # Remove exact duplicate triplets, but preserve different negatives for the same query.
    before = len(df)
    df = df.drop_duplicates(
        subset=["query", "positive_text", "negative_text"],
        keep="first",
    ).reset_index(drop=True)
    print(f"Exact duplicate triplets removed: {before - len(df)}")

    # Deterministic initial query ordering. Never use list(set(...)) for seeded sampling.
    unique_queries = sorted(df["query"].unique().tolist())
    print(f"Unique queries after cleaning: {len(unique_queries)}")

    # Deterministic query-level split.
    rng = random.Random(seed)
    shuffled_queries = unique_queries.copy()
    rng.shuffle(shuffled_queries)

    n_train_queries = int(len(shuffled_queries) * train_query_ratio)
    train_queries = shuffled_queries[:n_train_queries]
    val_queries = shuffled_queries[n_train_queries:]

    train_query_set = set(train_queries)
    val_query_set = set(val_queries)

    train_df = df[df["query"].isin(train_query_set)].copy()
    val_df = df[df["query"].isin(val_query_set)].copy()

    # Training-size ablation: sample complete queries, never individual triplets.
    if train_size < 1.0:
        n_selected = max(1, int(len(train_queries) * train_size))
        sample_rng = random.Random(seed)
        selected_queries = sample_rng.sample(sorted(train_queries), n_selected)
        selected_query_set = set(selected_queries)
        train_df = train_df[train_df["query"].isin(selected_query_set)].copy()
        print(f"Selected training queries: {len(selected_query_set)}")

    # Rename to the names expected by the training code.
    rename_map = {
        "query": QUERY_COLUMN,
        "positive_text": POSITIVE_COLUMN,
        "negative_text": NEGATIVE_COLUMN,
    }
    train_df = train_df.rename(columns=rename_map).reset_index(drop=True)
    val_df = val_df.rename(columns=rename_map).reset_index(drop=True)

    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

    # Final leakage check.
    final_train_queries = set(train_dataset[QUERY_COLUMN])
    final_val_queries = set(val_dataset[QUERY_COLUMN])
    overlap = final_train_queries & final_val_queries

    assert not overlap, (
        f"DATA LEAKAGE: {len(overlap)} queries occur in both train and validation."
    )

    print("\n================ DATASET SUMMARY ================")
    print(f"Training queries:      {len(final_train_queries)}")
    print(f"Validation queries:    {len(final_val_queries)}")
    print(f"Training triplets:     {len(train_dataset)}")
    print(f"Validation triplets:   {len(val_dataset)}")
    print(f"Query overlap:         {len(overlap)}")
    print("==================================================")

    print("\nTrain dataset:")
    print(train_dataset)
    print(train_dataset.column_names)

    print("\nValidation dataset:")
    print(val_dataset)
    print(val_dataset.column_names)

    print("\nFirst training example:")
    print(train_dataset[0])

    return train_dataset, val_dataset


train_dataset, val_dataset = build_bi_encoder_datasets(
    csv_path="stcalir_query_positive_negative_pairs.csv",
    train_query_ratio=TRAIN_QUERY_RATIO,
    train_size=TRAIN_SIZE,
    seed=SEED,
)


# Persist the exact cleaned/split datasets used by this run.
if SAVE_DATASETS:
    size_tag = str(int(TRAIN_SIZE * 100)) + "pct"
    split_dir = os.path.join(
        DATASET_OUTPUT_ROOT,
        f"train_{int(TRAIN_QUERY_RATIO * 100)}_val_{int((1 - TRAIN_QUERY_RATIO) * 100)}_{size_tag}"
    )
    train_path = os.path.join(split_dir, "train_dataset")
    val_path = os.path.join(split_dir, "val_dataset")
    os.makedirs(split_dir, exist_ok=True)

    train_dataset.save_to_disk(train_path)
    val_dataset.save_to_disk(val_path)

    print("\nSaved train dataset to:", train_path)
    print("Saved validation dataset to:", val_path)
print(train_dataset)
print(val_dataset)

Original CSV shape: (79110, 5)
Rows removed for missing/empty fields: 0
Exact duplicate triplets removed: 677
Unique queries after cleaning: 993

================ DATASET SUMMARY ================
Training queries:      893
Validation queries:    100
Training triplets:     70433
Validation triplets:   8000
Query overlap:         0

Train dataset:
Dataset({
    features: ['querytext', 'passagetext', 'negativepassagetext'],
    num_rows: 70433
})
['querytext', 'passagetext', 'negativepassagetext']

Validation dataset:
Dataset({
    features: ['querytext', 'passagetext', 'negativepassagetext'],
    num_rows: 8000
})
['querytext', 'passagetext', 'negativepassagetext']

First training example:
{'querytext': 'ما هي مدد سداد سعر التنازل عن العقارات السكنية والمهنية عند البيع بالتقسيط؟', 'passagetext': 'المادة 5: تتمم المادة 22 كما يلي: « يتم دفع سعر التنازل إما نقدا او بالتقسيط على مدى: خمس وعشرين (25) سنة بالنسبة للمحلات ذات الاستعمال السكني باستثناء الاملاك المشار إليها في المادة 2, الفقرتين

Saving the dataset (0/1 shards):   0%|          | 0/70433 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8000 [00:00<?, ? examples/s]


Saved train dataset to: ./stcalir_clean_splits/train_90_val_9_100pct/train_dataset
Saved validation dataset to: ./stcalir_clean_splits/train_90_val_9_100pct/val_dataset
Dataset({
    features: ['querytext', 'passagetext', 'negativepassagetext'],
    num_rows: 70433
})
Dataset({
    features: ['querytext', 'passagetext', 'negativepassagetext'],
    num_rows: 8000
})


In [21]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['querytext', 'passagetext', 'negativepassagetext'],
        num_rows: 70433
    })
    validation: Dataset({
        features: ['querytext', 'passagetext', 'negativepassagetext'],
        num_rows: 8000
    })
})


In [22]:
from huggingface_hub import login
access_token='your access token'
login(token=access_token)
dataset.push_to_hub("hatemestinbejaia/Arabic-Legal-IR-Training-Data-bi-encoder")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (3 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/hatemestinbejaia/Arabic-Legal-IR-Training-Data-bi-encoder/commit/c7bbc13335bd1de699950c2a599b75eb55b8b57e', commit_message='Upload dataset', commit_description='', oid='c7bbc13335bd1de699950c2a599b75eb55b8b57e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/hatemestinbejaia/Arabic-Legal-IR-Training-Data-bi-encoder', endpoint='https://huggingface.co', repo_type='dataset', repo_id='hatemestinbejaia/Arabic-Legal-IR-Training-Data-bi-encoder'), pr_revision=None, pr_num=None)

In [24]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split


def build_cross_encoder_dataset(
    csv_path,
    train_query_ratio=0.90,
    train_size=1.0,
    random_seed=42
):
    """
    Build query-level train/validation datasets for cross-encoder training.

    Each original row:
        query, positive_text, negative_text

    is converted into two cross-encoder examples:
        (query, positive_text, 1)
        (query, negative_text, 0)

    The split is performed BEFORE converting triplets into pairs,
    ensuring that no query occurs in both train and validation.

    train_size:
        Fraction of the training queries to use.
        Examples: 0.25, 0.50, 0.75, 1.0
    """

    # ---------------------------------------------------------
    # 1. Load data
    # ---------------------------------------------------------
    df = pd.read_csv(csv_path)

    # Keep only required columns
    df = df[
        ["query", "positive_text", "negative_text"]
    ].dropna()

    # ---------------------------------------------------------
    # 2. Get unique queries
    # ---------------------------------------------------------
    queries = df["query"].drop_duplicates().tolist()

    print(f"Total unique queries: {len(queries)}")

    # ---------------------------------------------------------
    # 3. Query-level train/validation split
    # ---------------------------------------------------------
    train_queries, val_queries = train_test_split(
        queries,
        test_size=1 - train_query_ratio,
        random_state=random_seed
    )

    train_queries = set(train_queries)
    val_queries = set(val_queries)

    # ---------------------------------------------------------
    # 4. Split original TRIPLETS by query
    # ---------------------------------------------------------
    train_triplets = df[df["query"].isin(train_queries)].copy()
    val_triplets = df[df["query"].isin(val_queries)].copy()

    print(f"Training queries:   {len(train_queries)}")
    print(f"Validation queries: {len(val_queries)}")

    print(f"Training triplets before size selection: "
          f"{len(train_triplets)}")

    print(f"Validation triplets: {len(val_triplets)}")

    # ---------------------------------------------------------
    # 5. Training-data size ablation
    # ---------------------------------------------------------
    # Select queries, NOT individual rows.
    #
    # This is important because all triplets belonging to a query
    # remain together.
    # ---------------------------------------------------------

    if train_size < 1.0:

        selected_train_queries = train_queries.copy()

        n_queries = int(
            len(selected_train_queries) * train_size
        )

        # Deterministic query sampling
        selected_train_queries = (
            pd.Series(list(selected_train_queries))
            .sample(
                n=n_queries,
                random_state=random_seed
            )
            .tolist()
        )

        selected_train_queries = set(selected_train_queries)

        train_triplets = train_triplets[
            train_triplets["query"].isin(
                selected_train_queries
            )
        ].copy()

        print(f"Training queries after size={train_size}: "
              f"{len(selected_train_queries)}")

    # ---------------------------------------------------------
    # 6. Convert triplets → cross-encoder pairs
    # ---------------------------------------------------------

    def triplets_to_pairs(triplets):

        positive = triplets[
            ["query", "positive_text"]
        ].copy()

        positive["label"] = 1

        positive = positive.rename(
            columns={
                "positive_text": "passage"
            }
        )

        negative = triplets[
            ["query", "negative_text"]
        ].copy()

        negative["label"] = 0

        negative = negative.rename(
            columns={
                "negative_text": "passage"
            }
        )

        pairs = pd.concat(
            [positive, negative],
            ignore_index=True
        )

        pairs = pairs.rename(
            columns={
                "query": "querytext",
                "passage": "passagetext"
            }
        )

        return pairs

    train_pairs = triplets_to_pairs(train_triplets)
    val_pairs = triplets_to_pairs(val_triplets)

    # ---------------------------------------------------------
    # 7. Convert to Hugging Face datasets
    # ---------------------------------------------------------

    train_dataset = Dataset.from_pandas(
        train_pairs,
        preserve_index=False
    )

    val_dataset = Dataset.from_pandas(
        val_pairs,
        preserve_index=False
    )

    # ---------------------------------------------------------
    # 8. Safety check: no query leakage
    # ---------------------------------------------------------

    train_query_set = set(train_dataset["querytext"])
    val_query_set = set(val_dataset["querytext"])

    overlap = train_query_set.intersection(
        val_query_set
    )

    print("\nLeakage check:")
    print(f"Train queries: {len(train_query_set)}")
    print(f"Validation queries: {len(val_query_set)}")
    print(f"Overlapping queries: {len(overlap)}")

    assert len(overlap) == 0, \
        "DATA LEAKAGE: queries occur in both train and validation!"

    print("✓ No query-level leakage.")

    return train_dataset, val_dataset

In [25]:
train_dataset, eval_dataset = build_cross_encoder_dataset(
    "stcalir_query_positive_negative_pairs.csv",
    train_query_ratio=0.90,
    train_size=1.0,
    random_seed=42
)
eval_dataset_raw = Dataset.from_dict({
    "querytext": eval_dataset["querytext"],
    "passagetext": eval_dataset["passagetext"],
    "labels": eval_dataset["label"],
})

print(train_dataset)
print(eval_dataset)

Total unique queries: 993
Training queries:   893
Validation queries: 100
Training triplets before size selection: 71409
Validation triplets: 7701

Leakage check:
Train queries: 893
Validation queries: 100
Overlapping queries: 0
✓ No query-level leakage.
Dataset({
    features: ['querytext', 'passagetext', 'label'],
    num_rows: 142818
})
Dataset({
    features: ['querytext', 'passagetext', 'label'],
    num_rows: 15402
})


In [26]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": train_dataset,
    "validation": eval_dataset
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['querytext', 'passagetext', 'label'],
        num_rows: 142818
    })
    validation: Dataset({
        features: ['querytext', 'passagetext', 'label'],
        num_rows: 15402
    })
})


In [27]:
from huggingface_hub import login
access_token='your access token'
login(token=access_token)
dataset.push_to_hub("hatemestinbejaia/Arabic-Legal-IR-Training-Data-cross-encoder")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (3 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/hatemestinbejaia/Arabic-Legal-IR-Training-Data-cross-encoder/commit/81fcb63d9a19f435e9360c0398bdf8c2000eb5ef', commit_message='Upload dataset', commit_description='', oid='81fcb63d9a19f435e9360c0398bdf8c2000eb5ef', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/hatemestinbejaia/Arabic-Legal-IR-Training-Data-cross-encoder', endpoint='https://huggingface.co', repo_type='dataset', repo_id='hatemestinbejaia/Arabic-Legal-IR-Training-Data-cross-encoder'), pr_revision=None, pr_num=None)